# 10 · Vizualizace SVM

Tento notebook je **edukační částí** cvičení 10. Prochází přednášku o metodě podpůrných vektorů: maximální okraj,
měkký okraj a parametr C, ztrátu pantu, jádra a parametr γ, škálování, regresi s ε-trubicí a jednotřídní SVM.
Každá metoda běží na malých syntetických datech a má graf s **ovládacími prvky** (posuvníky, nabídky, zaškrtávací pole). Kód nemusíte číst ani měnit.

**Jak pracovat:** spusťte všechny buňky (*Run → Run All Cells*). U každé části se nad grafem objeví ovládání; každá změna graf
překreslí. Kódové buňky jsou sbalené; pokud vás kód zajímá, rozbalíte ho kliknutím na lištu vlevo od buňky.
Každá část končí **✏️ Úkolem**; odpovědi vám pomohou v praktické části (`svm-practice-cz.ipynb`).

Pokud se ovládání nezobrazí, chybí balíček `ipywidgets` (viz README).

In [ ]:
import numpy as np                          # pole a výpočty
import pandas as pd                         # tabulky výsledků
import matplotlib.pyplot as plt             # grafy
import ipywidgets as widgets                # posuvníky, nabídky a zaškrtávací pole
from IPython.display import display         # tabulky
from sklearn.svm import SVC, SVR, OneClassSVM          # SVM pro klasifikaci, regresi a jednu třídu
from sklearn.model_selection import cross_val_score    # křížová validace
from sklearn.preprocessing import StandardScaler       # standardizace
from sklearn.pipeline import make_pipeline             # škálování a za ním model
from sklearn.datasets import make_moons, make_circles  # malé syntetické datové sady

BLUE, ORANGE, TEAL, GREY = "#0065BD", "#C2410C", "#00897B", "#8C96A0"  # barvy jako v přednášce
PALE_B, PALE_O = "#DCE9F6", "#F8E3D9"       # světlé oblasti tříd
plt.rcParams.update({"figure.figsize": (11, 4.2), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
STYLE = {"description_width": "190px"}      # místo pro popisky ovládání
WIDE = widgets.Layout(width="95%", max_width="560px")


def fslider(label, value, lo, hi, step, fmt=".2f"):
    return widgets.FloatSlider(value=value, min=lo, max=hi, step=step, description=label, readout_format=fmt,
                               continuous_update=False, style=STYLE, layout=WIDE)


def logslider(label, value, lo, hi):
    # posuvník v logaritmickém měřítku: lo a hi jsou exponenty desítky
    return widgets.FloatLogSlider(value=value, base=10, min=lo, max=hi, step=0.25, description=label,
                                  readout_format=".3g", continuous_update=False, style=STYLE, layout=WIDE)


def menu(label, options, value):
    return widgets.Dropdown(options=options, value=value, description=label, style=STYLE, layout=WIDE)


def check(label, value=False):
    return widgets.Checkbox(value=value, description=label, style=STYLE, layout=WIDE)


def controls(draw, **inputs):
    # zobrazí ovládání a po každé změně překreslí graf
    widgets.interact(draw, **inputs)


def show_classes(ax, X, y, s=40, alpha=1.0):
    # třída +1 modré kruhy, třída −1 oranžové trojúhelníky, jako v přednášce
    p = y == 1
    ax.scatter(X[p, 0], X[p, 1], s=s, color=BLUE, edgecolor="white", lw=0.6, alpha=alpha, zorder=3, label="třída +1")
    ax.scatter(X[~p, 0], X[~p, 1], s=s * 1.1, marker="^", color=ORANGE, edgecolor="white", lw=0.6, alpha=alpha, zorder=3, label="třída −1")


def ring(ax, P, color="black", s=200, label=None):
    ax.scatter(P[:, 0], P[:, 1], s=s, facecolors="none", edgecolors=color, lw=1.8, zorder=4, label=label)


def draw_regions(ax, model, xlim, ylim, margins=True, n=250):
    # oblasti tříd, hranice (skóre 0) a okraje (skóre ±1)
    xx, yy = np.meshgrid(np.linspace(*xlim, n), np.linspace(*ylim, n))
    zz = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-1e9, 0, 1e9], colors=[PALE_O, PALE_B], alpha=0.8)
    ax.contour(xx, yy, zz, levels=[0], colors="black", linewidths=2.2)
    if margins:
        ax.contour(xx, yy, zz, levels=[-1, 1], colors="black", linewidths=1.2, linestyles="--")


def blobs(seed, n, spread, c=1.3):
    g = np.random.default_rng(seed)
    X = np.r_[g.normal(c, spread, (n, 2)), g.normal(-c, spread, (n, 2))]
    return X, np.r_[np.ones(n), -np.ones(n)].astype(int)


print("Připraveno. Spusťte další buňky; pod každou částí se objeví ovládání.")

## 1. Perceptron a maximální okraj

**Cíl metody:** ze všech přímek bez trénovací chyby vybrat tu, která je nejdál od nejbližších bodů. Vzdálenost bodu od přímky
w·x + b = 0 je d = y·(w·x + b)/‖w‖; **okraj** je nejmenší d. Body, které ho určují, jsou **podpůrné vektory**.

Data jsou čtyři body z přednášek 09 a 10. Perceptron skončí u první přímky bez chyby, která závisí na pořadí vzorků.
SVM dá w = (0,8; 0,4), b = −1 a šířku okraje 2/‖w‖ = √5 ≈ 2,24 bez ohledu na pořadí.

**Ovládání:** pořadí vzorků pro perceptron; zaškrtávací pole přidá pátý bod třídy +1, jehož polohu měníte posuvníky.

In [ ]:
P1X = np.array([[3, 2], [0, -2], [1, 3], [-1, 2]], dtype=float)  # čtyři body z přednášky
P1Y = np.array([1, -1, 1, -1])


def perceptron(X, y, order, epochs=50):
    w, b = np.zeros(2), 0.0
    for _ in range(epochs):
        errors = 0
        for i in order:
            if y[i] * (w @ X[i] + b) <= 0:          # chyba: posun přímky ke vzorku
                w, b, errors = w + y[i] * X[i], b + y[i], errors + 1
        if errors == 0:
            break
    return w, b


def line(ax, w, b, xlim, **kw):
    xs = np.array(xlim, dtype=float)
    if abs(w[1]) > 1e-12:
        ax.plot(xs, -(w[0] * xs + b) / w[1], **kw)
    else:
        ax.axvline(-b / w[0], **kw)


def section1(order, add_point, px, py):
    X, y = P1X.copy(), P1Y.copy()
    if add_point:
        X, y = np.r_[X, [[px, py]]], np.r_[y, 1]
    order = list(order) + ([4] if add_point else [])
    w_p, b_p = perceptron(X, y, order)
    svm = SVC(kernel="linear", C=1e6).fit(X, y)     # velmi velké C = tvrdý okraj
    w_s, b_s = svm.coef_[0], svm.intercept_[0]
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))
    xl, yl = (-3, 5), (-4, 5)
    for ax, (w, b, title) in zip(axs, ((w_p, b_p, "perceptron"), (w_s, b_s, "SVM"))):
        d = y * (X @ w + b) / np.linalg.norm(w)
        if title == "SVM":
            draw_regions(ax, svm, xl, yl)
            ring(ax, X[svm.support_], label="podpůrné vektory")
        else:
            line(ax, w, b, xl, color=TEAL, lw=2.2)
        show_classes(ax, X, y, s=90)
        ax.set(xlim=xl, ylim=yl, xlabel="x₁", ylabel="x₂",
               title=f"{title}: w = ({w[0]:.2f}, {w[1]:.2f}), b = {b:.2f}\nnejmenší vzdálenost {d.min():.2f}")
        ax.set_aspect("equal")
    axs[1].legend(loc="lower right", fontsize=8)
    plt.show()
    errors = int((svm.predict(X) != y).sum())
    print(f"šířka okraje SVM 2/‖w‖ = {2 / np.linalg.norm(w_s):.3f}; "
          f"podpůrné vektory: {[f'x{i + 1}' for i in svm.support_]}")
    if errors:
        print(f"Přímka bez chyby neexistuje: SVM nechá {errors} bod(y) na špatné straně a perceptron se nezastaví (graf ukazuje jeho poslední přímku po 50 epochách).")
    if not add_point:
        assert np.allclose(w_s, [0.8, 0.4], atol=1e-3) and abs(b_s + 1) < 1e-3   # výsledek z přednášky


controls(section1,
         order=menu("pořadí pro perceptron", [("x₁, x₂, x₃, x₄", (0, 1, 2, 3)), ("x₂, x₁, x₃, x₄", (1, 0, 2, 3)),
                                                                ("x₃, x₁, x₂, x₄", (2, 0, 1, 3))], (0, 1, 2, 3)),
         add_point=check("přidat pátý bod (+1)"),
         px=fslider("pátý bod x₁", 0.5, -2, 4, 0.25), py=fslider("pátý bod x₂", 1.5, -3, 4, 0.25))

**✏️ Úkol 1**

* Měňte pořadí pro perceptron. Která přímka má větší nejmenší vzdálenost – některá z přímek perceptronu, nebo přímka SVM?
* Přidejte pátý bod do (3, 4), daleko za podpůrnými vektory. Změní se přímka SVM? A v bodě (0,5; 1,5)?
* Posuňte pátý bod do (−0,5; 1), mezi dva body třídy −1. Co se stane s okrajem? Pak ho posuňte do (−0,5; 0): co ukazuje hlášení pod grafem a panel perceptronu a proč?

## 2. Měkký okraj a parametr C

**Cíl metody:** dovolit bodům vstoupit do pásu nebo na špatnou stranu, ale každé porušení penalizovat: minimalizovat
½‖w‖² + C·Σξᵢ, kde volná proměnná ξᵢ = max(0, 1 − yᵢsᵢ). Malé C znamená široký pás a mnoho podpůrných vektorů; velké C úzký pás podle jednotlivých bodů.

**Ovládání:** C v logaritmickém měřítku, překryv tříd a semínko dat. Pod grafem: šířka okraje, počet podpůrných vektorů,
trénovací chyby a správnost při křížové validaci (5 částí).

In [ ]:
def section2(C, spread, seed):
    X, y = blobs(seed, 40, spread)
    m = SVC(kernel="linear", C=C).fit(X, y)
    xi = np.maximum(0, 1 - y * m.decision_function(X))   # volné proměnné
    fig, ax = plt.subplots(figsize=(6.2, 5.6))
    draw_regions(ax, m, (-4.5, 4.5), (-4.5, 4.5))
    show_classes(ax, X, y)
    ring(ax, X[(xi > 1e-6) & (xi < 1)], color=TEAL, label="0 < ξ < 1")
    ring(ax, X[xi >= 1], color=ORANGE, label="ξ ≥ 1")
    ax.set(xlim=(-4.5, 4.5), ylim=(-4.5, 4.5), xlabel="x₁", ylabel="x₂", title=f"C = {C:.3g}")
    ax.set_aspect("equal")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()
    cv = cross_val_score(SVC(kernel="linear", C=C), X, y, cv=5).mean()
    print(f"šířka okraje {2 / np.linalg.norm(m.coef_[0]):.2f} | podpůrné vektory {len(m.support_)} z {len(y)} | "
          f"trénovací chyby {int((m.predict(X) != y).sum())} | Σξ = {xi.sum():.1f} | správnost CV {cv:.3f}")


controls(section2, C=logslider("C", 1.0, -3, 3), spread=fslider("rozptyl tříd", 1.1, 0.5, 2.0, 0.1),
         seed=widgets.IntSlider(value=13, min=0, max=30, description="semínko", continuous_update=False, style=STYLE, layout=WIDE))

**✏️ Úkol 2**

* Posouvejte C od 0,001 do 1000. Jak se mění šířka okraje a počet podpůrných vektorů? Mění se stejně počet trénovacích chyb?
* Pro které C je správnost při křížové validaci nejlepší? Je to C s nejmenším počtem trénovacích chyb?
* Zmenšete rozptyl na 0,5 (oddělitelné třídy). Od jakého C je výsledek stejný jako s tvrdým okrajem?

## 3. Ztráta pantu a další ztráty

**Cíl:** porovnat, jak ztráty z přednášek 09 a 10 penalizují vzorek podle y·s (skóre se znaménkem třídy).
Ztráta pantu max(0, 1 − y·s) je nulová až od y·s ≥ 1; perceptronová max(0, −y·s) už od y·s > 0;
logaritmická log₂(1 + e^(−y·s)) není nikdy přesně nulová.

**Ovládání:** hodnota y·s jednoho vzorku; tabulka ukazuje pro něj tři ztráty.

In [ ]:
def section3(ys):
    t = np.linspace(-3, 3, 400)
    losses = {"pantu": lambda v: np.maximum(0, 1 - v), "perceptronová": lambda v: np.maximum(0, -v),
              "logaritmická": lambda v: np.log2(1 + np.exp(-v))}
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(t, (t <= 0).astype(float), color=GREY, label="počet chyb (0/1)")
    for (name, f), col, ls in zip(losses.items(), (BLUE, TEAL, ORANGE), ("-", "--", "-.")):
        ax.plot(t, f(t), color=col, ls=ls, lw=2.5, label=name)
        ax.scatter([ys], [f(ys)], color=col, s=60, zorder=4)
    ax.axvline(ys, color="black", lw=1, ls=":")
    ax.set(xlabel="y · s", ylabel="ztráta", ylim=(-0.1, 4))
    ax.legend(fontsize=9)
    plt.show()
    display(pd.DataFrame({name: [round(float(f(ys)), 3)] for name, f in losses.items()}, index=[f"y·s = {ys:g}"]))


controls(section3, ys=fslider("y · s", 0.5, -3, 3, 0.25))

**✏️ Úkol 3**

* Najděte hodnoty y·s, pro které je ztráta pantu 0; 0,5 a 2 (příklad P3 z přednášky). Co pro ně dává perceptronová ztráta?
* Vzorek s y·s = 0,5 je zařazen správně. Které ztráty ho přesto penalizují a proč to SVM chce?
* Pro y·s = −3: která ztráta roste nejrychleji? Porovnejte s kvadratickou chybou, která by pro vzdálenost 4 od cíle byla 16.

## 4. Jádra: polynomiální a RBF

**Cíl metody:** nahradit skalární součin jádrem K(x, z) a dostat nelineární hranici bez výpočtu nových příznaků.
Polynomiální jádro (γ·x·z + 1)^d dává hranici, která je polynomem stupně d; RBF jádro exp(−γ‖x − z‖²) dává hranici
libovolného tvaru a γ určuje, kam až sahá vliv jednoho podpůrného vektoru.

**Ovládání:** data (půlměsíce, kruhy, XOR), jádro, C, γ a stupeň d. Pod grafem: správnost při křížové validaci a počet podpůrných vektorů.

In [ ]:
def dataset(name, seed=4):
    if name == "moons":
        X, y = make_moons(n_samples=200, noise=0.2, random_state=seed)
    elif name == "circles":
        X, y = make_circles(n_samples=200, factor=0.45, noise=0.09, random_state=seed)
        X = X * 1.6
    else:                                              # XOR: čtyři mračna v rozích
        g = np.random.default_rng(seed)
        X = g.normal(0, 0.45, (200, 2)) + g.choice([-1, 1], (200, 2))
        y = (np.sign(X[:, 0]) != np.sign(X[:, 1])).astype(int)
    return X, np.where(y == 1, 1, -1)


def section4(data, kernel, C, gamma, degree):
    X, y = dataset(data)
    m = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree, coef0=1)
    acc = cross_val_score(m, X, y, cv=5).mean()
    m.fit(X, y)
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    lo, hi = X.min(0) - 0.5, X.max(0) + 0.5
    draw_regions(ax, m, (lo[0], hi[0]), (lo[1], hi[1]), margins=False)
    show_classes(ax, X, y, s=25)
    ring(ax, X[m.support_], s=70)
    ax.set(xlim=(lo[0], hi[0]), ylim=(lo[1], hi[1]), xlabel="x₁", ylabel="x₂",
           title=f"{kernel}, C = {C:.3g}, γ = {gamma:.3g}" + (f", d = {degree}" if kernel == "poly" else ""))
    plt.show()
    print(f"správnost CV {acc:.3f} | podpůrné vektory {len(m.support_)} z {len(y)}")


controls(section4, data=menu("data", [("půlměsíce", "moons"), ("kruhy", "circles"), ("XOR", "xor")], "moons"),
         kernel=menu("jádro", [("lineární", "linear"), ("polynomiální", "poly"), ("RBF", "rbf")], "rbf"),
         C=logslider("C", 1.0, -2, 3), gamma=logslider("γ", 1.0, -2, 2),
         degree=widgets.IntSlider(value=3, min=1, max=6, description="stupeň d", continuous_update=False, style=STYLE, layout=WIDE))

**✏️ Úkol 4**

* Pro každou datovou sadu porovnejte lineární, polynomiální (d = 2 a 3) a RBF jádro. Jaký stupeň stačí pro kruhy, jaký pro XOR a proč?
* RBF na půlměsících: posouvejte γ od 0,01 do 100 při C = 1. Kde je správnost nejlepší a jak vypadá hranice na obou koncích?
* Zvolte γ = 30. Dokážete výsledek zlepšit jen změnou C? Kterým směrem?

## 5. Škálování příznaků

**Cíl:** ukázat, proč SVM potřebuje příznaky ve srovnatelném měřítku. Okraj i RBF jádro měří vzdálenosti; příznak s velkým rozsahem v nich převáží.

**Ovládání:** násobek druhého příznaku (změna jednotek) a zapnutí standardizace. C = 1 a γ = 1 zůstávají pevné.

In [ ]:
def section5(factor, scale):
    X, y = dataset("moons", seed=7)
    X = X * np.array([1.0, factor])                    # stejná data v jiných jednotkách
    m = make_pipeline(StandardScaler(), SVC(C=1, gamma=1)) if scale else SVC(C=1, gamma=1)
    acc = cross_val_score(m, X, y, cv=5).mean()
    m.fit(X, y)
    fig, ax = plt.subplots(figsize=(6.2, 5))
    pad = (X.max(0) - X.min(0)) * 0.08
    xl, yl = (X[:, 0].min() - pad[0], X[:, 0].max() + pad[0]), (X[:, 1].min() - pad[1], X[:, 1].max() + pad[1])
    draw_regions(ax, m, xl, yl, margins=False)
    show_classes(ax, X, y, s=20)
    ax.set(xlim=xl, ylim=yl, xlabel="x₁", ylabel=f"x₂ × {factor:g}",
           title=("se standardizací" if scale else "bez škálování") + f" · správnost CV {acc:.3f}")
    plt.show()


controls(section5, factor=widgets.SelectionSlider(options=[0.01, 0.1, 1, 10, 30, 100], value=30, description="násobek x₂",
                                                  continuous_update=False, style=STYLE, layout=WIDE),
         scale=check("standardizovat příznaky"))

**✏️ Úkol 5**

* Bez škálování posouvejte násobek od 1 do 100. Jak se mění hranice a proč? Co se stane s násobkem 0,01?
* Zaškrtněte standardizaci. Záleží teď na násobku? Proč?
* Hodnoty C a γ byly vyladěny na datech v jedněch jednotkách. Lze je bez standardizace použít po změně jednotek?

## 6. Regrese s ε-trubicí (SVR)

**Cíl metody:** regresní funkce, která ignoruje odchylky menší než ε a větší penalizuje lineárně.
Body na hraně trubice nebo vně jsou podpůrné vektory. Šedý pás vpravo je mimo učební data:
tam je vidět, jak které jádro **extrapoluje**.

**Ovládání:** jádro, ε, C, γ a zobrazený rozsah.

In [ ]:
g6 = np.random.default_rng(3)
x6 = np.sort(g6.uniform(0, 5, 60))                    # učební vstupy od 0 do 5
y6 = 0.6 * x6 + np.sin(1.6 * x6) + g6.normal(0, 0.15, x6.size)   # trend plus vlna plus šum


def section6(kernel, eps, C, gamma, extend):
    m = SVR(kernel=kernel, epsilon=eps, C=C, gamma=gamma, degree=2, coef0=1).fit(x6.reshape(-1, 1), y6)
    t = np.linspace(0, 8 if extend else 5, 400)
    f = m.predict(t.reshape(-1, 1))
    fig, ax = plt.subplots(figsize=(9, 4.2))
    if extend:
        ax.axvspan(5, 8, color=GREY, alpha=0.15)
        ax.plot(t[t >= 5], 0.6 * t[t >= 5] + np.sin(1.6 * t[t >= 5]), color=GREY, ls=":", label="skutečná funkce")
    ax.fill_between(t, f - eps, f + eps, color=PALE_B, label="ε-trubice")
    ax.plot(t, f, color=BLUE, lw=2.5, label="model")
    ax.scatter(x6, y6, color=ORANGE, s=22, zorder=3)
    ax.scatter(x6[m.support_], y6[m.support_], s=90, facecolors="none", edgecolors="black", zorder=4, label="podpůrné vektory")
    ax.set(xlabel="x", ylabel="y", title=f"{kernel}, ε = {eps:g} · {len(m.support_)} podpůrných vektorů z {x6.size}")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()
    print(f"MAE na učebních datech {np.mean(np.abs(m.predict(x6.reshape(-1, 1)) - y6)):.3f}")


controls(section6, kernel=menu("jádro", [("RBF", "rbf"), ("lineární", "linear"), ("polynomiální d = 2", "poly")], "rbf"),
         eps=fslider("ε", 0.2, 0.0, 1.0, 0.05), C=logslider("C", 10.0, -1, 3), gamma=logslider("γ", 1.0, -2, 2),
         extend=check("zobrazit 5 až 8 (extrapolace)"))

**✏️ Úkol 6**

* Posouvejte ε od 0 do 1. Jak se mění počet podpůrných vektorů? Při jakém ε model přestane sledovat vlnu?
* Zaškrtněte rozsah extrapolace. Co předpovídá model s RBF jádrem v x = 8? A lineární? Který je blíž skutečné funkci a proč?
* Zlepší menší γ extrapolaci s RBF jádrem? Co to stojí uvnitř dat?

## 7. Jednotřídní SVM

**Cíl metody:** naučit hranici jen kolem běžných dat, bez označených odchylek. ν je přibližně horní mez podílu
učebních vzorků, které zůstanou vně; γ určuje, jak těsně hranice sleduje data.

**Ovládání:** ν, γ a zda učební data obsahují několik poruch. Křížky jsou nové odchylky, které model nikdy neviděl.

In [ ]:
g7 = np.random.default_rng(34)
normal7 = np.r_[g7.normal([0, 0], [1.0, 0.6], (150, 2)), g7.normal([2.6, 1.6], 0.5, (60, 2))]   # běžný provoz
new7 = g7.uniform([-4, -3.5], [5.5, 5], (40, 2))
new7 = new7[np.min(np.linalg.norm(new7[:, None] - normal7[None], axis=2), axis=1) > 0.9][:10]   # nové odchylky
dirty7 = g7.uniform([-3, -3], [4, 4], (12, 2))         # poruchy skryté v učebních datech


def section7(nu, gamma, dirty):
    train = np.r_[normal7, dirty7] if dirty else normal7
    m = OneClassSVM(nu=nu, gamma=gamma).fit(train)
    xx, yy = np.meshgrid(np.linspace(-4, 5.5, 250), np.linspace(-3.5, 5, 250))
    zz = m.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    out = m.predict(train) == -1
    found = int((m.predict(new7) == -1).sum())
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    ax.contourf(xx, yy, zz, levels=[0, zz.max()], colors=[PALE_B])
    ax.contour(xx, yy, zz, levels=[0], colors="black", linewidths=2)
    ax.scatter(*train[~out].T, s=16, color=BLUE, label="učební data")
    ax.scatter(*train[out].T, s=30, color=BLUE, edgecolor=ORANGE, lw=1.6, label="učební vzorek vně")
    ax.scatter(*new7.T, marker="x", s=90, color=ORANGE, lw=2.5, label="nové odchylky")
    ax.set(xlim=(-4, 5.5), ylim=(-3.5, 5), xlabel="x₁", ylabel="x₂",
           title=f"ν = {nu:g}, γ = {gamma:.3g} · {100 * out.mean():.0f} % vně · nalezeno {found}/{len(new7)}")
    ax.set_aspect("equal")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()


controls(section7, nu=fslider("ν", 0.05, 0.01, 0.5, 0.01), gamma=logslider("γ", 0.2, -2, 1),
         dirty=check("12 poruch v učebních datech"))

**✏️ Úkol 7**

* Posouvejte ν od 0,01 do 0,5. Porovnejte podíl učebních vzorků vně s ν. Kolik nových odchylek se najde na obou koncích?
* Zvyšte γ na 5. Jak vypadá hranice a co by se stalo s novými běžnými daty, která leží mezi učebními body?
* Zaškrtněte poruchy v učebních datech. Jak se změní hranice při ν = 0,02? Které ν pomůže a co to říká o čištění učebních dat?

## 8. Požehnání a prokletí dimenzionality

**Cíl:** ukázat, proč mapování do mnoha příznaků usnadní lineární oddělení – a proč to samo nestačí.
Vlevo n náhodných bodů v d rozměrech s **náhodnými** třídami; podíl přiřazení tříd, která nějaká nadrovina oddělí bez chyby.
Čára je Coverova věta P = 2^(1−n)·Σ_{k=0}^{d} C(n−1, k), body simulace (přesný test lineárním programem).
Vpravo 30 náhodných bodů ve 2D s náhodnými třídami a SVM s polynomiálním jádrem stupně p a velmi velkým C (prakticky tvrdý okraj).

**Ovládání:** rozměr d a počet bodů n (vlevo); stupeň polynomu p (vpravo). Pod grafem: učební správnost,
správnost při křížové validaci a počet podpůrných vektorů.

In [ ]:
from math import comb
from scipy.optimize import linprog                  # přesný test lineární oddělitelnosti


def cover(n, d):
    return 2 * sum(comb(n - 1, k) for k in range(d + 1)) / 2 ** n   # nadrovina s posunem v d rozměrech


def separable(X, y):
    s = np.where(y == 1, 1.0, -1.0)
    A = -s[:, None] * np.c_[X, np.ones(len(X))]      # najít w, b s y(w·x + b) ≥ 1
    return linprog(np.zeros(X.shape[1] + 1), A_ub=A, b_ub=-np.ones(len(X)), bounds=[(None, None)] * (X.shape[1] + 1), method="highs").status == 0


g8 = np.random.default_rng(8)
X8 = g8.uniform(-1, 1, (30, 2)); y8 = np.where(g8.random(30) < 0.5, 1, -1)   # náhodné body, náhodné třídy


def section8(d, n, p):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.4))
    ax = axs[0]
    ns = np.arange(1, 4 * (d + 1) + 1)
    ax.plot(ns, [cover(int(k), d) for k in ns], color=BLUE, label="Coverova věta")
    g = np.random.default_rng(d)
    sim = np.mean([separable(g.normal(size=(n, d)), np.where(g.random(n) < 0.5, 1, -1)) for _ in range(40)])
    ax.scatter([n], [sim], color=ORANGE, s=90, zorder=4, label=f"simulace n = {n}: {sim:.2f}")
    ax.axvline(2 * (d + 1), color=GREY, ls=":")
    ax.set(xlabel="počet bodů n", ylabel="podíl oddělitelných", ylim=(-0.05, 1.05), title=f"d = {d}")
    ax.legend(fontsize=9)
    ax = axs[1]
    m = SVC(kernel="poly", degree=p, coef0=1, gamma=1, C=1e6).fit(X8, y8)
    draw_regions(ax, m, (-1.1, 1.1), (-1.1, 1.1), margins=False)
    show_classes(ax, X8, y8, s=45)
    ax.set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), title=f"náhodné třídy, polynom stupně {p}")
    ax.set_aspect("equal")
    plt.show()
    train = m.score(X8, y8)
    cv = cross_val_score(SVC(kernel="poly", degree=p, coef0=1, gamma=1, C=1e6), X8, y8, cv=5).mean()
    print(f"učební správnost {train:.2f} | správnost CV {cv:.2f} | podpůrné vektory {len(m.support_)} z 30")


controls(section8, d=widgets.SelectionSlider(options=[1, 2, 5, 10, 20], value=2, description="d", continuous_update=False, style=STYLE, layout=WIDE),
         n=widgets.IntSlider(value=6, min=2, max=60, description="n", continuous_update=False, style=STYLE, layout=WIDE),
         p=widgets.IntSlider(value=1, min=1, max=8, description="stupeň p", continuous_update=False, style=STYLE, layout=WIDE))

**✏️ Úkol 8**

* Pro d = 2 najděte n, při kterém je simulovaný podíl asi polovina. Ověřte pravidlem n = 2(d + 1). Zopakujte pro d = 10.
* Vpravo zvyšujte stupeň p od 1 do 8. Od kterého stupně jsou všechny náhodné body zařazené správně? Jaká je pak správnost při křížové validaci a proč?
* Třídy jsou náhodné, a přesto je od určitého stupně učební chyba nulová. Co znamená nulová učební chyba v prostoru s mnoha příznaky a co je nutné ověřit kromě ní (přednáška: prokletí dimenzionality a okraj)?